# Phase 1.2 — Understanding EEG Signals

**EEG Sleep Stage Classification | Research Foundations**  
**Author / Maintainer:** Ben10-L  
**Status:** In progress — not yet assessed  
**Data policy:** All plots in this notebook use synthetic educational signals, **not real patient data**.

### Learning objectives
1. Distinguish neural action potentials from scalp EEG generation.
2. Explain EEG reference electrodes and bipolar derivations.
3. Visualize frequency bands and understand amplitude, frequency, and period.
4. Explain sampling, Nyquist frequency, and 30-second epochs.
5. Prepare channels-first tensors suitable for later deep-learning experiments.

Run all cells after installing the repository with `python -m pip install -e '.[dev]'`.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from eeg_sleep import (
    EEG_CHANNELS,
    FREQUENCY_BANDS,
    nyquist_frequency,
    sample_count,
    segment_epochs,
    simulate_sleep_stage,
    simulate_two_channel_eeg,
    sinusoid,
    time_vector,
)
from eeg_sleep.plotting import (
    plot_action_potential,
    plot_frequency_bands,
    plot_sampling,
    plot_stage_signatures,
    plot_two_channel_epoch,
)

SEED = 42
SAMPLING_RATE_HZ = 100
EPOCH_DURATION_S = 30
print("Phase 1.2 environment ready.")

## 1 — Biological foundations

A neuronal **action potential** is a rapid, propagating change in membrane potential. By contrast, scalp EEG is primarily generated by the combined, synchronized **postsynaptic currents** of populations of appropriately oriented cortical neurons.

> Do not confuse membrane potential in **millivolts (mV)** with scalp EEG amplitude often expressed in **microvolts (µV)**. The plot below is a stylized explanation of an action potential, **not an EEG trace or a Hodgkin–Huxley simulation**.

In [ ]:
fig = plot_action_potential()
plt.show()

## 2 — Physical foundations: channels and references

A recorded EEG channel is a difference between measured potentials. In Sleep-EDF Expanded, two relevant EEG derivations are **Fpz–Cz** and **Pz–Oz**. These are not independent measurements of a single electrode's absolute voltage.

\[
V_{Fpz-Cz}(t)=V_{Fpz}(t)-V_{Cz}(t)
\]
\[
V_{Pz-Oz}(t)=V_{Pz}(t)-V_{Oz}(t)
\]

The real Sleep-EDF channels are recorded at **100 Hz**. In this notebook we only generate synthetic signals with the same **array shape**, not actual signals from those electrodes.

In [ ]:
t_30s, synthetic_eeg = simulate_two_channel_eeg(
    sampling_rate_hz=SAMPLING_RATE_HZ,
    duration_seconds=EPOCH_DURATION_S,
    seed=SEED,
)
assert synthetic_eeg.shape == (2, 3000)
print("Channel names:", EEG_CHANNELS)
print("Time samples:", t_30s.shape)
print("EEG epoch tensor:", synthetic_eeg.shape)

fig = plot_two_channel_epoch()
plt.show()

## 3 — Oscillatory activity and frequency bands

Five commonly referenced frequency bands are delta, theta, alpha, beta, and gamma. Their boundaries are conventions and can vary between sources.

For a sinusoid:

\[
x(t)=A\sin(2\pi f t+\phi)
\]

where \(A\) is amplitude, \(f\) is frequency in hertz, and \(\phi\) is the phase in radians. The period is \(T=1/f\).

In [ ]:
for band in FREQUENCY_BANDS:
    print(f"{band.name:5s} | illustrative: {band.example_hz:>4.1f} Hz | period: {1000/band.example_hz:6.1f} ms")

fig = plot_frequency_bands()
plt.show()

### Experiment A — Compare two oscillations

Try changing the two frequencies and observing how many cycles appear in one second. Then change the amplitude to see how it affects the vertical axis.

In [ ]:
duration_s = 1.0
plotting_rate_hz = 500  # Dense plotting grid, not the Sleep-EDF acquisition rate.
t = time_vector(plotting_rate_hz, duration_s)
frequency_a_hz = 2.0
frequency_b_hz = 10.0
amplitude_uv = 20.0

wave_a = sinusoid(t, frequency_hz=frequency_a_hz, amplitude_uv=amplitude_uv)
wave_b = sinusoid(t, frequency_hz=frequency_b_hz, amplitude_uv=amplitude_uv)

fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
ax.plot(t, wave_a, label=f"{frequency_a_hz:g} Hz")
ax.plot(t, wave_b, label=f"{frequency_b_hz:g} Hz")
ax.set(xlabel="Time (seconds)", ylabel="Amplitude (µV)", title="Frequency comparison")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

print(f"Period at {frequency_a_hz:g} Hz: {1000/frequency_a_hz:.1f} ms")
print(f"Period at {frequency_b_hz:g} Hz: {1000/frequency_b_hz:.1f} ms")

## 4 — Different sleep stages are not just different frequencies

Sleep scoring considers both the timing and shape of characteristic patterns. Stage N2 commonly includes spindles and K-complexes; N3 involves prominent slow-wave activity. Similar frequencies can occur in different stages.

The following signals were **handcrafted solely to demonstrate visual features**. They cannot replace recorded EEG, expert-scored labels, or clinical staging criteria.

In [ ]:
fig = plot_stage_signatures()
plt.show()

# Example: retrieve an individual synthetic pattern.
t_n2, n2_example = simulate_sleep_stage("N2", seed=SEED)
print("Synthetic N2-like example shape:", n2_example.shape)

## 5 — Sampling a signal

A continuous waveform becomes a discrete numerical sequence after sampling. For sampling frequency \(f_s\):

\[
T_s=\frac{1}{f_s}, \qquad x[n]=x\!\left(\frac{n}{f_s}\right)
\]

The Nyquist frequency is \(f_N=f_s/2\). Frequencies above this limit are susceptible to aliasing. Real acquisition systems need **anti-aliasing filters**, not just a higher plotting resolution.

In [ ]:
fig = plot_sampling()
plt.show()

fs = SAMPLING_RATE_HZ
print(f"Sampling frequency: {fs} Hz")
print(f"Sampling interval: {1000/fs:.1f} ms")
print(f"Nyquist frequency: {nyquist_frequency(fs):.1f} Hz")

### Experiment B — Alias a high-frequency sinusoid

At a 100 Hz sampling rate, a 70 Hz sinusoid produces the same *sampled values* as a negative-frequency 30 Hz sinusoid, up to phase/sign convention. This illustrates **aliasing** and why proper analog anti-aliasing must happen before sampling.

In [ ]:
fs = 100
sample_times = time_vector(fs, 1.0)
sampled_70 = sinusoid(sample_times, frequency_hz=70, amplitude_uv=1)
sampled_minus_30 = sinusoid(sample_times, frequency_hz=30, amplitude_uv=-1)

print("Max absolute difference between samples:", np.max(np.abs(sampled_70 - sampled_minus_30)))
assert np.allclose(sampled_70, sampled_minus_30, atol=1e-12)

fig, ax = plt.subplots(figsize=(10, 3.3), layout="constrained")
ax.plot(sample_times[:35], sampled_70[:35], "o-", label="70 Hz sampled at 100 Hz", markersize=3)
ax.plot(sample_times[:35], sampled_minus_30[:35], "x--", label="-30 Hz equivalent samples", alpha=.7)
ax.set(xlabel="Time (s)", ylabel="Amplitude (a.u.)", title="Aliasing demonstration")
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 6 — From continuous EEG to 30-second epochs

At 100 Hz, a 30-second epoch contains **3,000 samples per channel**. Two channels give a channels-first matrix of shape **(2, 3000)**. A batch of 32 such epochs would have shape **(32, 2, 3000)**, following the input convention used by PyTorch `Conv1d`.

The example below segments synthetic signals only. In a real study, epochs must also match the timing of the expert sleep-stage annotations.

In [ ]:
n_samples = sample_count(SAMPLING_RATE_HZ, EPOCH_DURATION_S)
assert n_samples == 3000
print("Samples per channel:", n_samples)
print("Samples in a 2-channel epoch:", 2 * n_samples)
print("One epoch shape:", synthetic_eeg.shape)

# Demonstrate a 90-second continuous 2-channel simulation.
_, recording_90s = simulate_two_channel_eeg(
    sampling_rate_hz=100, duration_seconds=90, seed=SEED,
)
epochs = segment_epochs(
    recording_90s, sampling_rate_hz=100, epoch_duration_seconds=30,
)
assert epochs.shape == (3, 2, 3000)
print("90-second recording:", recording_90s.shape)
print("Epochs (epoch, channel, sample):", epochs.shape)

### Experiment C — Prepare an example batch

In a real pipeline each batch contains different labeled epochs. Here we make a small **shape demonstration only** from synthetic recordings — this is **not a training batch** with valid class labels.

In [ ]:
synthetic_batch = np.stack([
    simulate_two_channel_eeg(seed=seed)[1]
    for seed in range(32)
])

assert synthetic_batch.shape == (32, 2, 3000)
print("Example array shape:", synthetic_batch.shape)
print("Dtype:", synthetic_batch.dtype)
print("Number of numeric values:", synthetic_batch.size)

## 7 — Your coding exercises

1. Change the sinusoid frequencies to **3 Hz** and **12 Hz**. Compute the periods.
2. Re-run the two-channel epoch simulation at **128 Hz**. What is the new epoch shape?
3. Segment a **120-second** two-channel simulated recording into **30-second** epochs.
4. Explain in one paragraph why the waveform examples above **cannot** be used to report sleep-classification accuracy.
5. Run `python -m pytest -q` in your terminal and record the result in `docs/RESEARCH_LOG.md`.

**Do not mark this lesson complete** until you have explained the results in your own words and passed the knowledge quiz.

## Key references

- [PhysioNet — Sleep-EDF Expanded](https://physionet.org/content/sleep-edfx/1.0.0/)
- [MNE — Setting the EEG reference](https://mne.tools/stable/auto_tutorials/preprocessing/55_setting_eeg_reference.html)
- [The Scientific Basis of EEG](https://www.ncbi.nlm.nih.gov/books/NBK390351/)

**Next lesson:** Phase 1.3 — Filtering, artifacts, FFT, PSD, and spectrograms.